In [1]:
# ==========================================
# BINARY-AWARE RADIUS GAP PROJECT
# COMPLETE DATA RETRIEVAL + MERGE PIPELINE
# ==========================================

import pandas as pd
import numpy as np

from astroquery.vizier import Vizier
from astroquery.utils.tap.core import TapPlus

# ==========================================
# 1. LOAD SULLIVAN CATALOG
# ==========================================

print("\n===================================")
print("Loading Sullivan catalog...")
print("===================================")

Vizier.ROW_LIMIT = -1

tables = Vizier.get_catalogs("J/AJ/168/129")

# Table 0 = binary/star properties
sample_df = tables[0].to_pandas()

# Table 2 = planet properties
planet_df = tables[2].to_pandas()

print("Sample table rows:", len(sample_df))
print("Planet table rows:", len(planet_df))

# ==========================================
# 2. CLEAN KOI IDS
# ==========================================

sample_df["KOI"] = (
    sample_df["KOI"]
    .astype(str)
    .str.strip()
)

planet_df["KOI"] = (
    planet_df["KOI"]
    .astype(str)
    .str.strip()
)

planet_df["KOIpl"] = (
    planet_df["KOIpl"]
    .astype(str)
    .str.strip()
)

# ==========================================
# 3. MERGE SULLIVAN TABLES
# ==========================================

print("\n===================================")
print("Merging Sullivan tables...")
print("===================================")

binary_planet_df = planet_df.merge(
    sample_df,
    on="KOI",
    how="left"
)

print("Merged Sullivan rows:", len(binary_planet_df))

# ==========================================
# 4. QUERY NASA EXOPLANET ARCHIVE
# ==========================================

print("\n===================================")
print("Querying NASA Exoplanet Archive...")
print("===================================")

tap = TapPlus(
    url="https://exoplanetarchive.ipac.caltech.edu/TAP"
)

query = """
SELECT
    kepoi_name,
    koi_period,
    koi_period_err1,
    koi_period_err2,
    koi_dor,
    koi_dor_err1,
    koi_dor_err2,
    koi_duration,
    koi_duration_err1,
    koi_duration_err2,
    koi_impact,
    koi_impact_err1,
    koi_impact_err2,
    koi_depth,
    koi_depth_err1,
    koi_depth_err2,
    koi_steff,
    koi_srad,
    koi_smass
FROM q1_q17_dr25_koi
"""

job = tap.launch_job(query)

table = job.get_results()

exo_df = table.to_pandas()

print("NASA rows:", len(exo_df))

# ==========================================
# 5. BUILD MATCHING KEYS
# ==========================================

print("\n===================================")
print("Building matching keys...")
print("===================================")

# Sullivan:
# 42.01

binary_planet_df["KOIpl_clean"] = (
    binary_planet_df["KOIpl"]
    .astype(str)
    .str.lstrip("0")
    .str.strip()
)

# NASA:
# K00042.01 -> 42.01

exo_df["KOIpl_clean"] = (
    exo_df["kepoi_name"]
    .astype(str)
    .str.replace("K", "", regex=False)
    .str.lstrip("0")
    .str.strip()
)

print(
    binary_planet_df[
        ["KOIpl", "KOIpl_clean"]
    ].head()
)

print(
    exo_df[
        ["kepoi_name", "KOIpl_clean"]
    ].head()
)

# ==========================================
# 6. MERGE NASA + SULLIVAN
# ==========================================

print("\n===================================")
print("Merging NASA + Sullivan catalogs...")
print("===================================")

master_df = binary_planet_df.merge(
    exo_df,
    on="KOIpl_clean",
    how="left"
)

print("Final merged rows:", len(master_df))

# ==========================================
# 7. MATCH SUCCESS CHECK
# ==========================================

print("\n===================================")
print("MATCH STATISTICS")
print("===================================")

print(
    "Matched planets:",
    master_df["kepoi_name"].notnull().sum()
)

print(
    "Missing planets:",
    master_df["kepoi_name"].isnull().sum()
)

# ==========================================
# 8. DERIVED SECONDARY STELLAR RADIUS
# ==========================================

print("\n===================================")
print("Computing stellar properties...")
print("===================================")

master_df["Rstar2"] = (
    master_df["Rstar1"]
    * master_df["Rstar2/Rstar1"]
)

# ==========================================
# 9. APPROXIMATE STELLAR MASSES
# ==========================================

# Main-sequence approximation:
# M ~ R^1.25

master_df["Mstar1_est"] = (
    master_df["Rstar1"] ** 1.25
)

master_df["Mstar2_est"] = (
    master_df["Rstar2"] ** 1.25
)

# ==========================================
# 10. STELLAR DENSITIES
# ==========================================

# rho ~ M / R^3
# in solar-density units

master_df["rho_pri"] = (
    master_df["Mstar1_est"]
    / (master_df["Rstar1"] ** 3)
)

master_df["rho_sec"] = (
    master_df["Mstar2_est"]
    / (master_df["Rstar2"] ** 3)
)

# ==========================================
# 11. TRANSIT-DERIVED DENSITY
# ==========================================

print("\n===================================")
print("Computing transit-derived densities...")
print("===================================")

# Constants
G = 6.67430e-11
pi = np.pi

# Solar density in SI
rho_sun_si = 1408  # kg/m^3

# Convert period days -> seconds
P_sec = master_df["koi_period"] * 86400

# a/Rstar
a_over_R = master_df["koi_dor"]

# SI density
rho_si = (
    (3 * pi / (G * P_sec**2))
    * (a_over_R**3)
)

# Convert to solar-density units
master_df["rho_transit"] = (
    rho_si / rho_sun_si
)

# ==========================================
# 12. TRANSIT DENSITY UNCERTAINTIES
# ==========================================

print("\n===================================")
print("Computing uncertainties...")
print("===================================")

master_df["rho_transit_fracerr"] = (
    3
    * (
        np.abs(master_df["koi_dor_err1"])
        / master_df["koi_dor"]
    )
)

master_df["rho_transit_err"] = (
    master_df["rho_transit"]
    * master_df["rho_transit_fracerr"]
)

# ==========================================
# 13. STELLAR DENSITY UNCERTAINTIES
# ==========================================

# Simple approximation
master_df["rho_pri_err"] = (
    0.15 * master_df["rho_pri"]
)

master_df["rho_sec_err"] = (
    0.15 * master_df["rho_sec"]
)

# ==========================================
# 14. HOST LIKELIHOODS
# ==========================================

print("\n===================================")
print("Computing host likelihoods...")
print("===================================")

var_pri = (
    master_df["rho_transit_err"]**2
    + master_df["rho_pri_err"]**2
)

var_sec = (
    master_df["rho_transit_err"]**2
    + master_df["rho_sec_err"]**2
)

master_df["lnL_pri"] = (
    -0.5
    * (
        (
            master_df["rho_transit"]
            - master_df["rho_pri"]
        )**2
        / var_pri
    )
)

master_df["lnL_sec"] = (
    -0.5
    * (
        (
            master_df["rho_transit"]
            - master_df["rho_sec"]
        )**2
        / var_sec
    )
)

# ==========================================
# 15. PRIORS
# ==========================================

master_df["prior_pri"] = 0.5
master_df["prior_sec"] = 0.5

# ==========================================
# 16. HOST POSTERIOR PROBABILITIES
# ==========================================

print("\n===================================")
print("Computing posterior probabilities...")
print("===================================")

L_pri = (
    np.exp(master_df["lnL_pri"])
    * master_df["prior_pri"]
)

L_sec = (
    np.exp(master_df["lnL_sec"])
    * master_df["prior_sec"]
)

norm = L_pri + L_sec

master_df["p_pri"] = L_pri / norm
master_df["p_sec"] = L_sec / norm

# ==========================================
# 17. QUALITY FLAGS
# ==========================================

needed_cols = [
    "koi_period",
    "koi_dor",
    "Rppri",
    "Rpsec",
    "rho_pri",
    "rho_sec",
    "rho_transit",
    "p_pri",
    "p_sec"
]

master_df["usable"] = (
    master_df[needed_cols]
    .notnull()
    .all(axis=1)
)

print("\n===================================")
print("USABLE SAMPLE")
print("===================================")

print(
    "Usable planets:",
    master_df["usable"].sum()
)

# ==========================================
# 18. FINAL ANALYSIS TABLE
# ==========================================

print("\n===================================")
print("Building analysis-ready table...")
print("===================================")

final_cols = [

    # IDs
    "KOI",
    "KOIpl",
    "kepoi_name",

    # Binary properties
    "Sep",

    # Stellar properties
    "Teff1",
    "Teff2",
    "Rstar1",
    "Rstar2",
    "Mstar1_est",
    "Mstar2_est",

    # Planet radii
    "Rppri",
    "Rpsec",

    # Transit observables
    "koi_period",
    "koi_dor",
    "koi_duration",
    "koi_impact",

    # Densities
    "rho_transit",
    "rho_transit_err",
    "rho_pri",
    "rho_sec",

    # Host probabilities
    "p_pri",
    "p_sec",

    # Quality
    "usable"
]

analysis_df = master_df[final_cols]

# ==========================================
# 19. SAVE FILES
# ==========================================

print("\n===================================")
print("Saving files...")
print("===================================")

master_df.to_csv(
    "binary_radius_gap_master_catalog.csv",
    index=False
)

analysis_df.to_csv(
    "binary_radius_gap_analysis_ready.csv",
    index=False
)

usable_df = analysis_df[
    analysis_df["usable"] == True
]

usable_df.to_csv(
    "binary_radius_gap_usable_sample.csv",
    index=False
)

print("Saved:")
print("1. binary_radius_gap_master_catalog.csv")
print("2. binary_radius_gap_analysis_ready.csv")
print("3. binary_radius_gap_usable_sample.csv")

# ==========================================
# 20. FINAL SUMMARY
# ==========================================

print("\n===================================")
print("FINAL SUMMARY")
print("===================================")

print("Total Sullivan planets:", len(master_df))

print(
    "Matched NASA planets:",
    master_df["kepoi_name"].notnull().sum()
)

print(
    "Usable planets:",
    master_df["usable"].sum()
)

print("\nExample usable rows:")

print(
    usable_df.head(10)
)


Loading Sullivan catalog...
Sample table rows: 286
Planet table rows: 404

Merging Sullivan tables...
Merged Sullivan rows: 404

Querying NASA Exoplanet Archive...
NASA rows: 2000

Building matching keys...
    KOIpl KOIpl_clean
0   42.01       42.01
1  112.01      112.01
2  112.02      112.02
3  162.01      162.01
4  163.01      163.01
  kepoi_name KOIpl_clean
0  K00753.01      753.01
1  K00754.01      754.01
2  K00755.01      755.01
3  K00756.01      756.01
4  K00756.02      756.02

Merging NASA + Sullivan catalogs...
Final merged rows: 404

MATCH STATISTICS
Matched planets: 142
Missing planets: 262

Computing stellar properties...

Computing transit-derived densities...

Computing uncertainties...

Computing host likelihoods...

Computing posterior probabilities...

USABLE SAMPLE
Usable planets: 142

Building analysis-ready table...

Saving files...
Saved:
1. binary_radius_gap_master_catalog.csv
2. binary_radius_gap_analysis_ready.csv
3. binary_radius_gap_usable_sample.csv

FINAL S

In [2]:
# ==========================================
# SAVE CLEAN ANALYSIS INPUT FILE
# FOR FUTURE ISOCHRONE FITTING
# ==========================================

import pandas as pd
import numpy as np

# ------------------------------------------
# LOAD CURRENT MASTER FILE
# ------------------------------------------

master_df = pd.read_csv(
    "binary_radius_gap_master_catalog.csv"
)

# ------------------------------------------
# KEEP ONLY USABLE SYSTEMS
# ------------------------------------------

usable_df = master_df[
    master_df["usable"] == True
].copy()

print("Usable planets:", len(usable_df))

# ==========================================
# BUILD CLEAN STELLAR+PLANET TABLE
# ==========================================

# This file will become the central input
# for:
#
# - isochrone fitting
# - host probability refinement
# - population inference
# - future hierarchical modeling
#
# Keep:
# - measured quantities
# - derived quantities
# - uncertainties
#
# Avoid:
# - temporary/debug columns
# ==========================================

analysis_cols = [

    # --------------------------------------
    # IDs
    # --------------------------------------

    "KOI",
    "KOIpl",
    "kepoi_name",

    # --------------------------------------
    # Binary properties
    # --------------------------------------

    "Sep",

    "dmi",
    "e_dmi",

    "dmK",
    "e_dmK",

    # --------------------------------------
    # Stellar properties
    # PRIMARY
    # --------------------------------------

    "Teff1",
    "E_Teff1",
    "e_Teff1",

    "Rstar1",
    "E_Rstar1",
    "e_Rstar1",

    # --------------------------------------
    # Stellar properties
    # SECONDARY
    # --------------------------------------

    "Teff2",
    "E_Teff2",
    "e_Teff2",

    "Rstar2",
    "Rstar2/Rstar1",
    "E_Rstar2/Rstar1",
    "e_Rstar2/Rstar1",

    # --------------------------------------
    # Current approximate masses
    # (temporary until isochrones)
    # --------------------------------------

    "Mstar1_est",
    "Mstar2_est",

    # --------------------------------------
    # Planet radii
    # --------------------------------------

    "Rppri",
    "E_Rppri",
    "e_Rppri",

    "Rpsec",
    "E_Rpsec",
    "e_Rpsec",

    # --------------------------------------
    # Transit observables
    # --------------------------------------

    "koi_period",
    "koi_period_err1",
    "koi_period_err2",

    "koi_dor",
    "koi_dor_err1",
    "koi_dor_err2",

    "koi_duration",
    "koi_duration_err1",
    "koi_duration_err2",

    "koi_impact",
    "koi_impact_err1",
    "koi_impact_err2",

    "koi_depth",
    "koi_depth_err1",
    "koi_depth_err2",

    # --------------------------------------
    # Transit-derived density
    # --------------------------------------

    "rho_transit",
    "rho_transit_err",

    # --------------------------------------
    # Current stellar density estimates
    # --------------------------------------

    "rho_pri",
    "rho_sec",

    # --------------------------------------
    # Current host probabilities
    # --------------------------------------

    "p_pri",
    "p_sec",

]

# ==========================================
# BUILD CLEAN DATAFRAME
# ==========================================

clean_df = usable_df[analysis_cols].copy()

# ==========================================
# BASIC QUALITY CHECKS
# ==========================================

print("\n===================================")
print("QUALITY CHECK")
print("===================================")

print(clean_df.isnull().sum())

# ==========================================
# SAVE FILES
# ==========================================

# Main CSV
clean_df.to_csv(
    "binary_radius_gap_clean_input.csv",
    index=False
)

# Also save parquet (better for large analysis)
clean_df.to_parquet(
    "binary_radius_gap_clean_input.parquet",
    index=False
)

print("\n===================================")
print("FILES SAVED")
print("===================================")

print("1. binary_radius_gap_clean_input.csv")
print("2. binary_radius_gap_clean_input.parquet")

# ==========================================
# SUMMARY
# ==========================================

print("\n===================================")
print("FINAL SUMMARY")
print("===================================")

print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))

print("\nColumns:")
print(clean_df.columns.tolist())

print("\nHead:")
print(clean_df.head())


Usable planets: 142

QUALITY CHECK
KOI                    0
KOIpl                  0
kepoi_name             0
Sep                    0
dmi                  119
e_dmi                119
dmK                   18
e_dmK                 18
Teff1                  0
E_Teff1                0
e_Teff1                0
Rstar1                 0
E_Rstar1               0
e_Rstar1               0
Teff2                  0
E_Teff2                0
e_Teff2                0
Rstar2                 0
Rstar2/Rstar1          0
E_Rstar2/Rstar1        0
e_Rstar2/Rstar1        0
Mstar1_est             0
Mstar2_est             0
Rppri                  0
E_Rppri                0
e_Rppri                0
Rpsec                  0
E_Rpsec                0
e_Rpsec                0
koi_period             0
koi_period_err1        0
koi_period_err2        0
koi_dor                0
koi_dor_err1           0
koi_dor_err2           0
koi_duration           0
koi_duration_err1      0
koi_duration_err2      0
koi_impact     